## Export model and artifacts

### Purpose

This notebook simulates what a customer might produce when training models outside of Databricks.  
It generates example model artifacts and writes them to a Unity Catalog volume so they can be ingested, logged, and deployed using the BYOM workflow.

Use this notebook for:
- Demonstrations
- Template validation
- End-to-end testing of the pipeline

Skip this notebook in real engagements where customer artifacts already exist.

### What This Notebook Produces

The following artifacts are written to a Unity Catalog volume:

- Native framework models (XGBoost, scikit-learn, PyTorch)
- PyFunc models with preprocessing and postprocessing logic
- Canonical input sample (`canonical_input.json`)
- `checksums.json` for artifact integrity validation

These artifacts simulate externally trained model outputs.


In [0]:
%pip install -q threadpoolctl>=3.5.0 xgboost torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [ ]:

# Widget parameters for job orchestration
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("volume_name", "volume", "Volume Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
volume = dbutils.widgets.get("volume_name")

### Artifact layout

Defines the expected directory and file structure inside the Unity Catalog volume.  
Subsequent logging notebooks assume this structure when loading artifacts.

In [0]:
import os
import json
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import xgboost as xgb
import joblib
import numpy as np
import hashlib
import pandas as pd

# UC volume path
artifact_root = f"/Volumes/{catalog}/{schema}/{volume}"

# Table names (example: Iris; replace with your own for other data)
input_table_name = "iris"
features_table_name = "iris_features"
labels_table_name = "iris_labels"
# Registered model name for DL PyFunc (written to artifacts.json)
dl_model_name = "iris_torch_pyfunc"

# XGBoost (native + PyFunc) artifact paths
xgb_model_path = os.path.join(artifact_root, "xgb.pkl")
sklearn_model_path = os.path.join(artifact_root, "sklearn_model.pkl")
pyfunc_xgb_model_path = os.path.join(artifact_root, "pyfunc_xgb.pkl")
pyfunc_preproc_path = os.path.join(artifact_root, "pyfunc_preproc.json")
pyfunc_postproc_path = os.path.join(artifact_root, "pyfunc_postproc.json")

# Shared: canonical input (all model types)
canonical_input_path = os.path.join(artifact_root, "canonical_input.json")

# PyTorch (deep learning): single contract (artifacts.json + torch_model.pt)
artifacts_dl_path = os.path.join(artifact_root, "artifacts.json")
torch_model_path = os.path.join(artifact_root, "torch_model.pt")
checksums_path = os.path.join(artifact_root, "checksums.json")

# Single list for checksums (and optional verify); add new artifacts here
ALL_ARTIFACTS = [
    ("xgb.pkl", xgb_model_path),
    ("sklearn_model.pkl", sklearn_model_path),
    ("pyfunc_xgb.pkl", pyfunc_xgb_model_path),
    ("pyfunc_preproc.json", pyfunc_preproc_path),
    ("pyfunc_postproc.json", pyfunc_postproc_path),
    ("canonical_input.json", canonical_input_path),
    ("artifacts.json", artifacts_dl_path),
    ("torch_model.pt", torch_model_path),
]

### Create sample input and feature tables

Generates sample data to:
- Train example models
- Provide evaluation inputs
- Demonstrate inference flows

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

dataset = load_iris(as_frame=True)
pdf = dataset.data.copy()
pdf["target"] = dataset.target
pdf.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace("_cm", "") for c in pdf.columns]
pdf["id"] = np.arange(len(pdf), dtype=np.int64)

pdf["sepal_length_div_sepal_width"] = pdf["sepal_length"] / pdf["sepal_width"]
pdf["petal_length_div_petal_width"] = pdf["petal_length"] / pdf["petal_width"].replace(0, np.nan)
pdf["sepal_area_approx"] = pdf["sepal_length"] * pdf["sepal_width"]
pdf["petal_area_approx"] = pdf["petal_length"] * pdf["petal_width"]
pdf["length_sum"] = pdf["sepal_length"] + pdf["petal_length"]
pdf["width_sum"] = pdf["sepal_width"] + pdf["petal_width"]
pdf["sepal_petal_length_diff"] = pdf["sepal_length"] - pdf["petal_length"]
pdf["sepal_petal_width_diff"] = pdf["sepal_width"] - pdf["petal_width"]
pdf["sepal_length_x_petal_length"] = pdf["sepal_length"] * pdf["petal_length"]
pdf["sepal_width_x_petal_width"] = pdf["sepal_width"] * pdf["petal_width"]
pdf["sepal_length_bin"] = pd.cut(pdf["sepal_length"], bins=[4.0, 5.0, 6.0, 8.0], labels=False).astype(float)

feature_columns = [c for c in pdf.columns if c != "target"]
features_pdf = pdf[feature_columns]
labels_pdf = pdf[["id", "target"]]

spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{input_table_name}")
spark.createDataFrame(features_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{features_table_name}")
spark.createDataFrame(labels_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{labels_table_name}")

### Generate model artifacts

Trains sample models and exports:

- Native serialized models (`.pkl`, `.pt`, etc.)
- PyFunc wrappers with configuration files
- Supporting metadata required for MLflow logging

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

def write_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

# Load data (shared by all model types)
features_sdf = spark.read.table(f"{catalog}.{schema}.{features_table_name}")
labels_sdf = spark.read.table(f"{catalog}.{schema}.{labels_table_name}")
X = features_sdf.drop("id").toPandas()
Y = labels_sdf.drop("id").toPandas()
feature_columns = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, stratify=Y, random_state=42
)
y_train_np = y_train.values.ravel()
y_test_np = y_test.values.ravel()

# 1) XGBoost: train and export (native + PyFunc)
print("Training XGBoost (native + PyFunc)...")
xgb_clf = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4,
    subsample=0.9, colsample_bytree=0.9, random_state=42
)
xgb_clf.fit(X_train, y_train_np)

joblib.dump(xgb_clf, xgb_model_path)
joblib.dump(xgb_clf, pyfunc_xgb_model_path)

write_json(pyfunc_preproc_path, {
    "center": True, "means": X_train.mean().tolist(),
    "scale": False, "stds": [1.0] * X_train.shape[1]
})
write_json(pyfunc_postproc_path, {"output": "class"})

canonical_input = X.iloc[[0]].copy()
canonical_input.to_json(canonical_input_path, orient="records")
print("  ✓ XGBoost + PyFunc artifacts saved")

# 1b) scikit-learn: train and export (native MLflow flavor)
print("Training scikit-learn (native)...")
from sklearn.ensemble import RandomForestClassifier
sk_clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
sk_clf.fit(X_train, y_train_np)
joblib.dump(sk_clf, sklearn_model_path)
print("  ✓ sklearn_model.pkl saved")

# 2) PyTorch: train and export (deep learning)
print("Training PyTorch (deep learning)...")
scaler_dl = StandardScaler()
X_train_scaled = scaler_dl.fit_transform(X_train)
X_test_scaled = scaler_dl.transform(X_test)
X_train_t = torch.from_numpy(X_train_scaled.astype(np.float32))
y_train_t = torch.from_numpy(y_train_np)
X_test_t = torch.from_numpy(X_test_scaled.astype(np.float32))
y_test_t = torch.from_numpy(y_test_np)

input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = len(np.unique(y_train_np))

class SimpleClassifierNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim), nn.ReLU(),
            nn.Linear(hid_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

pytorch_model = SimpleClassifierNet(input_dim, hidden_dim, output_dim)
optimizer = optim.Adam(pytorch_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
pytorch_model.train()
for epoch in range(100):
    optimizer.zero_grad()
    loss = criterion(pytorch_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

pytorch_model.eval()
with torch.no_grad():
    _, pred = torch.max(pytorch_model(X_test_t), 1)
    dl_accuracy = (pred == y_test_t).float().mean().item()
print(f"  PyTorch test accuracy: {dl_accuracy:.4f}")

meta = {
    "feature_columns": feature_columns,
    "input_dim": input_dim, "hidden_dim": hidden_dim, "output_dim": output_dim,
    "accuracy": dl_accuracy,
    "class_names": ["setosa", "versicolor", "virginica"]
}
torch.save({"model": pytorch_model, "scaler": scaler_dl, "meta": meta}, torch_model_path)
write_json(artifacts_dl_path, {
    "model_name": dl_model_name,
    "dependencies": ["torch", "pandas", "numpy", "scikit-learn", "joblib"],
    "feature_columns": feature_columns,
    "example_input": canonical_input.values.tolist(),
    "torch_model_path": "torch_model.pt",
})
print("  ✓ PyTorch artifacts saved (artifacts.json + torch_model.pt)")


### Generate and verify checksums

Creates `checksums.json` to validate artifact integrity before logging and registration.
Checksum validation is optional but recommended for production use cases.

In [ ]:
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

checksums = {name: sha256_file(path) for name, path in ALL_ARTIFACTS}
with open(checksums_path, "w", encoding="utf-8") as f:
    json.dump(checksums, f, indent=2)
print("checksums.json updated")

print(f"\nContents of {artifact_root}:\n")
for a in sorted(dbutils.fs.ls(artifact_root), key=lambda x: x.name):
    print(f"  {a.name:<40} {a.size/1024:>10.2f} KB")
print("\n✓ Next: Notebook 1 (log/register) → Notebook 2 (deploy)")